In [30]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import json
from datetime import datetime
from tqdm import tqdm
import hydra
import numpy as np
import cv2
from PIL import Image
import lovely_tensors as lt
import imageio as iio
lt.monkey_patch()
import torch

device = 'cuda:0'

In [ ]:
from hmr4d.utils.body_model import BodyModelSMPLX
from hmr4d.utils.vis.renderer import Renderer
from hmr4d.utils.video_io_utils import read_video_np, get_writer
from hmr4d.utils.vis.cv2_utils import draw_kpts_with_conf_batch, draw_kpts_batch

smplx = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="neutral", num_pca_comps=12, flat_hand_mean=False,
).to(device)

In [40]:
root = Path("inputs/uni3c_aligned")
random = True
n_els = -1

# _load_dataset
indices = sorted([str(x.relative_to(root)) for x in root.glob("*/*")])
if random:
    np.random.seed(42)
    np.random.shuffle(indices)

# _get_idx2meta
n_els = len(indices) if n_els == -1 else n_els
idx2meta = {k: v for k, v in enumerate(indices[:n_els])}

In [25]:
# _load_data
idx = 25
mid = idx2meta[idx]
batch = torch.load(root / f"{mid}/batch_meta.pt")

/tmp/ipykernel_367481/2138764128.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  batch = torch.load(root / f"{mid}/batch_meta.pt")


In [ ]:
renderer_c = Renderer(batch['meta']['width'], batch['meta']['height'], device="cuda", 
                      faces=smplx.faces, K=batch['K_fullimg'][0])

verts = smplx(**{k:v.to(device) for k,v in batch['smpl_params_c'].items()}).vertices
images = read_video_np(root / f"{mid}/final.mp4", start_frame=1)  # (T, H, W, 3), np.uint8

writer1 = get_writer('tmp.mp4', fps=30, crf=23)
for j in tqdm(range(120), desc=f"Rendering Global"):
    # black_backg = np.zeros((batch['meta']['height'], batch['meta']['width'], 3)).astype(np.uint8)
    backg = cv2.resize(images[j], (batch['meta']['width'], batch['meta']['height']))

    smpl_rgbs, smpl_depths = renderer_c.render_mesh(
        verts[j], background=backg, return_depth=True
    )
    
    writer1.write_frame(smpl_rgbs)
writer1.close()

Rendering Global: 100%|██████████| 120/120 [00:02<00:00, 40.94it/s]
